# Two-Head Model Evaluation (test.ipynb)
xenium_train_2head.ipynb의 검증/평가 부분을 분리한 테스트 노트북.  
학습된 Two-Head 모델을 로드하여 검증 데이터에 대한 추론, 메트릭 계산, 시각화를 수행합니다.

In [ ]:
# ===== 1. 설정 및 라이브러리 임포트 =====
import os
import random
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ===== 2. 마커 유전자 및 하이퍼파라미터 정의 =====
data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/Xenium/'

NUM_GENES = 19
BATCH_SIZE = 16
NUM_WORKERS = 4
ENCODER_NAME = 'tu-convnext_tiny'

MARKER_GENES = [
    # Epithelial
    "EPCAM", "EGFR",
    # Fibroblast / Stromal
    "ACTA2", "PDGFRA", "PDGFRB", "SFRP4",
    # Endothelial
    "PECAM1",
    # Macrophage / Myeloid
    "CD68", "AIF1", "FCGR3A", "MRC1",
    # T cell
    "CD3E", "CD4", "CD8A", "TRAC",
    # B cell
    "CD79A", "MS4A1", "BANK1", "TCL1A"
]

In [ ]:
# ===== 3. 데이터셋 클래스 및 데이터 로딩 =====
class MultiScaleRegressionDataset(Dataset):
    def __init__(self, img_20x_list, img_5x_list, label_prop_list, label_int_list, augment=False):
        self.img_20x_paths = img_20x_list
        self.img_5x_paths = img_5x_list
        self.label_prop_paths = label_prop_list
        self.label_int_paths = label_int_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.img_20x_paths)

    def __getitem__(self, idx):
        img_20x = np.array(Image.open(self.img_20x_paths[idx]).convert('RGB'))
        img_5x = np.array(Image.open(self.img_5x_paths[idx]).convert('RGB'))
        label_prop = np.load(self.label_prop_paths[idx]).astype(np.float32)
        label_int = np.load(self.label_int_paths[idx]).astype(np.float32)

        label_prop = np.nan_to_num(label_prop, nan=0.0, posinf=1.0, neginf=0.0)
        label_int = np.nan_to_num(label_int, nan=0.0, posinf=1.0, neginf=0.0)
        label_prop = np.clip(label_prop, 0.0, 1.0)
        label_int = np.clip(label_int, 0.0, 1.0)

        label_prop = torch.from_numpy(label_prop)
        label_int = torch.from_numpy(label_int)

        img_20x = self.to_tensor(img_20x.copy())
        img_5x = self.to_tensor(img_5x.copy())

        return img_20x, img_5x, label_prop, label_int


# 파일 리스트 구성
img_20x_list = sorted(glob(f'{data_dir}/image/Xenium/**/*.tiff', recursive=True))
img_5x_list = [p.replace('/image/', '/image_5x/') for p in img_20x_list]
label_prop_list = [p.replace('/image/', '/label_proportion/').replace('.tiff', '.npy') for p in img_20x_list]
label_int_list = [p.replace('/image/', '/label_intensity/').replace('.tiff', '.npy') for p in img_20x_list]

valid_idx = [i for i in range(len(img_20x_list))
             if os.path.exists(img_5x_list[i])
             and os.path.exists(label_prop_list[i])
             and os.path.exists(label_int_list[i])]
print(f'Valid samples: {len(valid_idx)} / {len(img_20x_list)}')

img_20x_list = [img_20x_list[i] for i in valid_idx]
img_5x_list = [img_5x_list[i] for i in valid_idx]
label_prop_list = [label_prop_list[i] for i in valid_idx]
label_int_list = [label_int_list[i] for i in valid_idx]

# Train / Val split (학습과 동일한 seed 사용)
indices = list(range(len(img_20x_list)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Val samples: {len(val_idx)}')

val_dataset = MultiScaleRegressionDataset(
    [img_20x_list[i] for i in val_idx], [img_5x_list[i] for i in val_idx],
    [label_prop_list[i] for i in val_idx], [label_int_list[i] for i in val_idx],
    augment=False
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
print(f'Val batches: {len(val_loader)}')

In [ ]:
# ===== 4. 모델 정의 및 체크포인트 로드 =====
class MultiScaleRegressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=15):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')
        self.encoder_5x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')

        enc_channels = self.encoder_20x.out_channels
        feat_dim = enc_channels[-1]

        self.head_prop = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
        )
        self.head_int = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
            nn.Sigmoid()
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)
        feat_5x = self.encoder_5x(x_5x)
        fused = feat_20x[-1] + feat_5x[-1]
        pooled = F.adaptive_avg_pool2d(fused, 1).flatten(1)
        prop_logits = self.head_prop(pooled)
        intensity = self.head_int(pooled)
        return prop_logits, intensity


model = MultiScaleRegressionModel(ENCODER_NAME, NUM_GENES).to(device)

ckpt_path = os.path.join(model_path, 'train_2head_best.pt')
assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict, strict=False)
model.eval()
print(f'Checkpoint loaded: {ckpt_path}')

total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')

In [ ]:
# ===== 5. 평가 메트릭 함수 정의 =====
def pearson_corr_vector(pred, target):
    """Per-gene Pearson correlation. pred, target: (B, G) -> returns (G,)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return pcc

In [ ]:
# ===== 6. 검증 데이터 추론 (Inference) =====
all_pred_prop, all_gt_prop = [], []
all_pred_int, all_gt_int = [], []

with torch.no_grad():
    for img_20x, img_5x, gt_prop, gt_int in tqdm(val_loader, desc='Evaluation', leave=True):
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        pred_prop_logits, pred_int = model(img_20x, img_5x)
        all_pred_prop.append(torch.sigmoid(pred_prop_logits).cpu())
        all_gt_prop.append(gt_prop)
        all_pred_int.append(pred_int.cpu())
        all_gt_int.append(gt_int)

all_pred_prop = torch.cat(all_pred_prop)
all_gt_prop = torch.cat(all_gt_prop)
all_pred_int = torch.cat(all_pred_int)
all_gt_int = torch.cat(all_gt_int)

pcc_prop = pearson_corr_vector(all_pred_prop, all_gt_prop)
pcc_int = pearson_corr_vector(all_pred_int, all_gt_int)
mae_prop = (all_pred_prop - all_gt_prop).abs().mean(dim=0)
mae_int = (all_pred_int - all_gt_int).abs().mean(dim=0)

print(f'Inference complete: {len(all_gt_prop)} samples')

In [ ]:
# ===== 7. 유전자별 PCC & MAE 테이블 출력 =====
print(f'\n{"Gene":>10s} | {"PCC(prop)":>10s} {"MAE(prop)":>10s} | {"PCC(int)":>10s} {"MAE(int)":>10s}')
print('-' * 60)
for gi, gene in enumerate(MARKER_GENES):
    print(f'{gene:>10s} | {pcc_prop[gi]:10.4f} {mae_prop[gi]:10.4f} | {pcc_int[gi]:10.4f} {mae_int[gi]:10.4f}')
print('-' * 60)
print(f'{"Mean":>10s} | {pcc_prop.mean():10.4f} {mae_prop.mean():10.4f} | {pcc_int.mean():10.4f} {mae_int.mean():10.4f}')

In [ ]:
# ===== 8. 유전자별 PCC & MAE 막대 차트 =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(NUM_GENES)
w = 0.35

ax = axes[0]
ax.bar(x - w/2, pcc_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, pcc_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Per-Gene Pearson Correlation')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(x - w/2, mae_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, mae_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('Per-Gene Mean Absolute Error')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 9. Scatter Plot: Pred vs GT (Proportion & Intensity) =====
fig, axes = plt.subplots(4, 5, figsize=(25, 15))
fig.suptitle('Proportion: Predicted vs Ground Truth', fontsize=16, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_prop[:, gi].numpy()
    pred_vals = all_pred_prop[:, gi].numpy()
    ax.scatter(gt_vals, pred_vals, alpha=0.15, s=8, color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{gene}\nPCC={pcc_prop[gi]:.3f}', fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 5, figsize=(25, 15))
fig.suptitle('Intensity: Predicted vs Ground Truth', fontsize=16, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_int[:, gi].numpy()
    pred_vals = all_pred_int[:, gi].numpy()
    ax.scatter(gt_vals, pred_vals, alpha=0.15, s=8, color='coral')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, alpha=0.7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{gene}\nPCC={pcc_int[gi]:.3f}', fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 10. 오차 분포 Boxplot =====
error_prop = (all_pred_prop - all_gt_prop).numpy()
error_int = (all_pred_int - all_gt_int).numpy()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
bp = ax.boxplot([error_prop[:, gi] for gi in range(NUM_GENES)],
                labels=MARKER_GENES, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linewidth=0.8, linestyle='--')
ax.set_ylabel('Prediction Error (Pred - GT)')
ax.set_title('Proportion Error Distribution')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
bp = ax.boxplot([error_int[:, gi] for gi in range(NUM_GENES)],
                labels=MARKER_GENES, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('coral')
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linewidth=0.8, linestyle='--')
ax.set_ylabel('Prediction Error (Pred - GT)')
ax.set_title('Intensity Error Distribution')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 11. 유전자 그룹별 성능 요약 =====
gene_groups = {
    'Epithelial/Tumor': [0, 1, 2, 3],
    'Stromal/Fibroblast': [4, 5, 6, 7],
    'T Cell': [8, 9, 10],
    'B Cell': [11, 12],
    'Macrophage': [13],
    'Endothelial': [14],
}

group_names = list(gene_groups.keys())
group_pcc_prop = [pcc_prop[idx].mean().item() for idx in gene_groups.values()]
group_pcc_int = [pcc_int[idx].mean().item() for idx in gene_groups.values()]
group_mae_prop = [mae_prop[idx].mean().item() for idx in gene_groups.values()]
group_mae_int = [mae_int[idx].mean().item() for idx in gene_groups.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gx = np.arange(len(group_names))
gw = 0.35

ax = axes[0]
ax.bar(gx - gw/2, group_pcc_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_pcc_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean PCC'); ax.set_title('PCC by Gene Group')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(gx - gw/2, group_mae_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_mae_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean MAE'); ax.set_title('MAE by Gene Group')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 12. 랜덤 샘플 히트맵: GT vs Pred =====
np.random.seed(42)
n_samples = 20
sample_idx = np.random.choice(len(all_gt_prop), n_samples, replace=False)
sample_idx = np.sort(sample_idx)

fig, axes = plt.subplots(2, 2, figsize=(22, 10))

im = axes[0, 0].imshow(all_gt_prop[sample_idx].numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 0].set_yticks(range(NUM_GENES)); axes[0, 0].set_yticklabels(MARKER_GENES, fontsize=8)
axes[0, 0].set_xlabel('Sample Index'); axes[0, 0].set_title('Proportion — Ground Truth')
plt.colorbar(im, ax=axes[0, 0], fraction=0.02)

im = axes[0, 1].imshow(all_pred_prop[sample_idx].numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 1].set_yticks(range(NUM_GENES)); axes[0, 1].set_yticklabels(MARKER_GENES, fontsize=8)
axes[0, 1].set_xlabel('Sample Index'); axes[0, 1].set_title('Proportion — Prediction')
plt.colorbar(im, ax=axes[0, 1], fraction=0.02)

im = axes[1, 0].imshow(all_gt_int[sample_idx].numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
axes[1, 0].set_yticks(range(NUM_GENES)); axes[1, 0].set_yticklabels(MARKER_GENES, fontsize=8)
axes[1, 0].set_xlabel('Sample Index'); axes[1, 0].set_title('Intensity — Ground Truth')
plt.colorbar(im, ax=axes[1, 0], fraction=0.02)

im = axes[1, 1].imshow(all_pred_int[sample_idx].numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
axes[1, 1].set_yticks(range(NUM_GENES)); axes[1, 1].set_yticklabels(MARKER_GENES, fontsize=8)
axes[1, 1].set_xlabel('Sample Index'); axes[1, 1].set_title('Intensity — Prediction')
plt.colorbar(im, ax=axes[1, 1], fraction=0.02)

plt.suptitle('Random 20 Samples: GT vs Prediction Heatmap', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 13. 유전자별 분포 오버레이 (GT vs Pred) =====
fig, axes = plt.subplots(4, 5, figsize=(25, 12))
fig.suptitle('Proportion Distribution: GT (blue) vs Pred (red)', fontsize=14, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_prop[:, gi].numpy()
    pred_vals = all_pred_prop[:, gi].numpy()
    ax.hist(gt_vals, bins=50, alpha=0.5, color='steelblue', label='GT', density=True)
    ax.hist(pred_vals, bins=50, alpha=0.5, color='crimson', label='Pred', density=True)
    ax.set_title(f'{gene}', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 5, figsize=(25, 12))
fig.suptitle('Intensity Distribution: GT (orange) vs Pred (red)', fontsize=14, y=1.01)
for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    gt_vals = all_gt_int[:, gi].numpy()
    pred_vals = all_pred_int[:, gi].numpy()
    ax.hist(gt_vals, bins=50, alpha=0.5, color='orange', label='GT', density=True)
    ax.hist(pred_vals, bins=50, alpha=0.5, color='crimson', label='Pred', density=True)
    ax.set_title(f'{gene}', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 14. 개별 샘플 막대 차트: GT vs Pred =====
np.random.seed(77)
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
fig.suptitle('Individual Sample Comparison: GT vs Pred', fontsize=14, y=1.02)
show_idx = np.random.choice(len(all_gt_prop), n_show, replace=False)

for col, si in enumerate(show_idx):
    x = np.arange(NUM_GENES)
    ax = axes[0, col]
    ax.barh(x - 0.2, all_gt_prop[si].numpy(), 0.4, label='GT', color='steelblue', alpha=0.7)
    ax.barh(x + 0.2, all_pred_prop[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title(f'Sample {si}\n(Proportion)', fontsize=9)
    ax.legend(fontsize=6); ax.grid(True, alpha=0.3, axis='x')

    ax = axes[1, col]
    ax.barh(x - 0.2, all_gt_int[si].numpy(), 0.4, label='GT', color='orange', alpha=0.7)
    ax.barh(x + 0.2, all_pred_int[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=7)
    ax.set_xlim(0, 1); ax.set_title(f'Sample {si}\n(Intensity)', fontsize=9)
    ax.legend(fontsize=6); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 15. 전체 평균 비교: GT Mean vs Pred Mean =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(NUM_GENES)
w = 0.35

ax = axes[0]
ax.bar(x - w/2, all_gt_prop.mean(dim=0).numpy(), w, label='GT Mean', color='steelblue')
ax.bar(x + w/2, all_pred_prop.mean(dim=0).numpy(), w, label='Pred Mean', color='crimson')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('Mean Value'); ax.set_title('Proportion: GT Mean vs Pred Mean')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(x - w/2, all_gt_int.mean(dim=0).numpy(), w, label='GT Mean', color='orange')
ax.bar(x + w/2, all_pred_int.mean(dim=0).numpy(), w, label='Pred Mean', color='crimson')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('Mean Value'); ax.set_title('Intensity: GT Mean vs Pred Mean')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 16. 패치 이미지 + GT vs Pred 비교 시각화 =====
np.random.seed(123)
n_show = 8
show_idx = np.random.choice(len(all_gt_prop), n_show, replace=False)

fig, axes = plt.subplots(n_show, 4, figsize=(28, 5 * n_show),
                         gridspec_kw={'width_ratios': [1, 1, 1.5, 1.5]})
fig.suptitle('Patch Image + GT vs Pred Comparison', fontsize=18, y=1.005)

for row, si in enumerate(show_idx):
    img_20x = np.array(Image.open(val_dataset.img_20x_paths[si]).convert('RGB'))
    ax = axes[row, 0]
    ax.imshow(img_20x)
    ax.set_title(f'Sample {si} — 20x Patch', fontsize=10)
    ax.axis('off')

    img_5x = np.array(Image.open(val_dataset.img_5x_paths[si]).convert('RGB'))
    ax = axes[row, 1]
    ax.imshow(img_5x)
    ax.set_title(f'Sample {si} — 5x Context', fontsize=10)
    ax.axis('off')

    x = np.arange(NUM_GENES)
    ax = axes[row, 2]
    ax.barh(x - 0.2, all_gt_prop[si].numpy(), 0.4, label='GT', color='steelblue', alpha=0.7)
    ax.barh(x + 0.2, all_pred_prop[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=8)
    ax.set_xlim(0, 1); ax.set_title('Proportion', fontsize=10)
    ax.legend(fontsize=7, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

    ax = axes[row, 3]
    ax.barh(x - 0.2, all_gt_int[si].numpy(), 0.4, label='GT', color='orange', alpha=0.7)
    ax.barh(x + 0.2, all_pred_int[si].numpy(), 0.4, label='Pred', color='crimson', alpha=0.7)
    ax.set_yticks(x); ax.set_yticklabels(MARKER_GENES, fontsize=8)
    ax.set_xlim(0, 1); ax.set_title('Intensity', fontsize=10)
    ax.legend(fontsize=7, loc='lower right'); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 17. Global PCC 분석 (scipy 기반) =====
pred_prop_np = all_pred_prop.numpy()
gt_prop_np = all_gt_prop.numpy()
pred_int_np = all_pred_int.numpy()
gt_int_np = all_gt_int.numpy()

# --- Per-Gene Global PCC (scipy) ---
gene_pcc_prop_scipy = []
gene_pcc_int_scipy = []
for gi in range(NUM_GENES):
    r_prop, _ = stats.pearsonr(pred_prop_np[:, gi], gt_prop_np[:, gi])
    r_int, _ = stats.pearsonr(pred_int_np[:, gi], gt_int_np[:, gi])
    gene_pcc_prop_scipy.append(r_prop)
    gene_pcc_int_scipy.append(r_int)

# --- Per-Sample Global PCC ---
sample_pcc_prop = []
sample_pcc_int = []
for si in range(len(gt_prop_np)):
    r_prop, _ = stats.pearsonr(pred_prop_np[si], gt_prop_np[si])
    r_int, _ = stats.pearsonr(pred_int_np[si], gt_int_np[si])
    sample_pcc_prop.append(r_prop)
    sample_pcc_int.append(r_int)
sample_pcc_prop = np.array(sample_pcc_prop)
sample_pcc_int = np.array(sample_pcc_int)

# --- Global PCC (전체 flatten) ---
global_pcc_prop, _ = stats.pearsonr(pred_prop_np.flatten(), gt_prop_np.flatten())
global_pcc_int, _ = stats.pearsonr(pred_int_np.flatten(), gt_int_np.flatten())

# --- 결과 출력 ---
print('=' * 70)
print('Global PCC Analysis (scipy.stats.pearsonr)')
print('=' * 70)

print(f'\n{"Gene":>10s} | {"PCC(prop)":>10s} | {"PCC(int)":>10s}')
print('-' * 40)
for gi, gene in enumerate(MARKER_GENES):
    print(f'{gene:>10s} | {gene_pcc_prop_scipy[gi]:10.4f} | {gene_pcc_int_scipy[gi]:10.4f}')
print('-' * 40)
print(f'{"Mean":>10s} | {np.mean(gene_pcc_prop_scipy):10.4f} | {np.mean(gene_pcc_int_scipy):10.4f}')

print(f'\nPer-Sample PCC (mean ± std):')
print(f'  Proportion: {sample_pcc_prop.mean():.4f} ± {sample_pcc_prop.std():.4f}')
print(f'  Intensity:  {sample_pcc_int.mean():.4f} ± {sample_pcc_int.std():.4f}')

print(f'\nGlobal PCC (all values flattened):')
print(f'  Proportion: {global_pcc_prop:.4f}')
print(f'  Intensity:  {global_pcc_int:.4f}')

# --- 시각화 (2×3 subplot) ---
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
fig.suptitle('Global PCC Analysis', fontsize=16, y=1.02)

# (0,0) Per-Gene PCC Bar Chart
x = np.arange(NUM_GENES)
w = 0.35
ax = axes[0, 0]
ax.bar(x - w/2, gene_pcc_prop_scipy, w, label='Proportion', color='steelblue')
ax.bar(x + w/2, gene_pcc_int_scipy, w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('PCC (scipy)'); ax.set_title('Per-Gene Global PCC')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

# (0,1) Per-Sample PCC Distribution (Proportion)
ax = axes[0, 1]
ax.hist(sample_pcc_prop, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=sample_pcc_prop.mean(), color='red', linewidth=2, linestyle='--',
           label=f'Mean={sample_pcc_prop.mean():.3f}')
ax.set_xlabel('PCC'); ax.set_ylabel('Count')
ax.set_title('Per-Sample PCC Distribution (Proportion)')
ax.legend(); ax.grid(True, alpha=0.3)

# (0,2) Per-Sample PCC Distribution (Intensity)
ax = axes[0, 2]
ax.hist(sample_pcc_int, bins=50, color='coral', alpha=0.7, edgecolor='white')
ax.axvline(x=sample_pcc_int.mean(), color='red', linewidth=2, linestyle='--',
           label=f'Mean={sample_pcc_int.mean():.3f}')
ax.set_xlabel('PCC'); ax.set_ylabel('Count')
ax.set_title('Per-Sample PCC Distribution (Intensity)')
ax.legend(); ax.grid(True, alpha=0.3)

# (1,0) Global Scatter (Proportion, flattened)
ax = axes[1, 0]
subsample = np.random.choice(len(pred_prop_np.flatten()), min(5000, len(pred_prop_np.flatten())), replace=False)
ax.scatter(gt_prop_np.flatten()[subsample], pred_prop_np.flatten()[subsample],
           alpha=0.1, s=4, color='steelblue')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1)
ax.set_xlabel('GT'); ax.set_ylabel('Pred')
ax.set_title(f'Global Scatter — Proportion\nPCC={global_pcc_prop:.4f}')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# (1,1) Global Scatter (Intensity, flattened)
ax = axes[1, 1]
subsample = np.random.choice(len(pred_int_np.flatten()), min(5000, len(pred_int_np.flatten())), replace=False)
ax.scatter(gt_int_np.flatten()[subsample], pred_int_np.flatten()[subsample],
           alpha=0.1, s=4, color='coral')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1)
ax.set_xlabel('GT'); ax.set_ylabel('Pred')
ax.set_title(f'Global Scatter — Intensity\nPCC={global_pcc_int:.4f}')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# (1,2) Summary Table
ax = axes[1, 2]
ax.axis('off')
summary_data = [
    ['Metric', 'Proportion', 'Intensity'],
    ['Per-Gene PCC (mean)', f'{np.mean(gene_pcc_prop_scipy):.4f}', f'{np.mean(gene_pcc_int_scipy):.4f}'],
    ['Per-Sample PCC (mean)', f'{sample_pcc_prop.mean():.4f}', f'{sample_pcc_int.mean():.4f}'],
    ['Per-Sample PCC (std)', f'{sample_pcc_prop.std():.4f}', f'{sample_pcc_int.std():.4f}'],
    ['Global PCC (flatten)', f'{global_pcc_prop:.4f}', f'{global_pcc_int:.4f}'],
]
table = ax.table(cellText=summary_data[1:], colLabels=summary_data[0],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)
ax.set_title('Summary', fontsize=14, pad=20)

plt.tight_layout()
plt.show()